# Train the next-step fire spread models (Colab, A100)

Trains and evaluates three models on the sample shards from `firespread build-samples`:

| mode | inputs | role |
|---|---|---|
| `ml` | weather, fuels, terrain, moisture, burned area | ML-only baseline |
| `physics_features` | + Rothermel spread rate, direction, intensity | ML with physics features |
| `hybrid` | + calibrated physics forecast (`phys_prob`), residual on top | hybrid (needs the spread engine output) |

Each step predicts which still-unburned 360 m blocks burn in the next 12 hours. Scores: IoU of the burned
area at t+1 and IoU of the **newly burned** area (persistence scores 0 there).

**Before running:** push the repo to GitHub (the notebook installs `firespread` from `main`) and upload
`fire_samples.tar.gz` to `My Drive/fire_spread/`. Runtime: A100 GPU, standard RAM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/fire_spread"
DATA = "/content/data"
RUN = "run_01"  # change for each new set of runs
OUT = f"{DRIVE}/results/{RUN}"

In [ ]:
# firespread without its geospatial dependencies (only the ML part is used here)
!pip install -q "zarr>=3" "xarray>=2025.1"
!pip install -q --no-deps --force-reinstall "git+https://github.com/sharmeen-k/fire_spread_modeling@main"

In [ ]:
import os
import tarfile
import time

if not os.path.exists(f"{DATA}/split.csv"):
    t = time.time()
    os.makedirs(DATA, exist_ok=True)
    with tarfile.open(f"{DRIVE}/fire_samples.tar.gz") as tf:
        tf.extractall(DATA, filter="data")
    print(f"extracted in {time.time() - t:.0f} s")
print(len(os.listdir(f"{DATA}/samples")), "fire shards")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from firespread import ml

fires = ml.load_shards(Path(DATA) / "samples", Path(DATA) / "split.csv")
summary = pd.DataFrame(
    [
        dict(
            fire=f.fire_id,
            split=f.split,
            samples=f.x.shape[0],
            blocks=f.valid.sum(),
            grid=f"{f.valid.shape[1]}x{f.valid.shape[0]}",
        )
        for f in fires
    ]
)
print(summary.groupby("split").agg(fires=("fire", "count"), samples=("samples", "sum")))
print("channels:", len(fires[0].channels), "| phys_prob available:", ml.PHYS_PROB in fires[0].channels)
summary.sort_values(["split", "fire"])

## Train

`epochs`, `samples_per_epoch` (random 64x64 tiles centred on the fire front) and `base` (U-Net width) are the
main knobs. On an A100 one run of 40 epochs should take on the order of 10-20 minutes. The best epoch on the
validation fires (pooled new-burn IoU) is kept.

In [ ]:
cfg_common = dict(epochs=40, batch=32, tile=64, lr=2e-3, base=32, samples_per_epoch=4000, seed=42)
results = {}
for mode in ["ml", "physics_features"]:
    print(f"=== {mode} ===")
    results[mode] = ml.train(fires, ml.TrainConfig(mode=mode, **cfg_common), Path(OUT))

In [ ]:
# hybrid: only once the spread engine has written phys_prob into the shards
if ml.PHYS_PROB in fires[0].channels:
    print("=== hybrid ===")
    results["hybrid"] = ml.train(fires, ml.TrainConfig(mode="hybrid", **cfg_common), Path(OUT))
else:
    print("phys_prob not in the shards yet; skipping the hybrid")

## Results

In [ ]:
rows = []
for mode, r in results.items():
    for split in ("val", "test"):
        if split in r:
            rows.append(dict(model=mode, split=split, **r[split]))
            rows.append(dict(model="persistence", split=split, **r[split + "_persistence"]))
table = pd.DataFrame(rows).drop_duplicates(["model", "split"]).set_index(["split", "model"]).sort_index()
table.to_csv(f"{OUT}/summary.csv")
table.round(3)

In [ ]:
# per test fire
per = []
for mode in results:
    p = Path(OUT) / f"{mode}_test_steps.csv"
    if p.exists():
        d = pd.read_csv(p)
        for fid, g in d.groupby("fire"):
            gg = g[g.new_obs > 0]
            per.append(
                dict(
                    model=mode,
                    fire=fid,
                    iou=g.iou.mean(),
                    iou_new_pooled=gg.tp_new.sum() / max(gg.union_new.sum(), 1),
                )
            )
pd.DataFrame(per).pivot(index="fire", columns="model").round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for mode, r in results.items():
    h = pd.DataFrame(r["history"])
    ax[0].plot(h.epoch, h.loss, label=mode)
    ax[1].plot(h.epoch, h.val_iou_new_pooled, label=mode)
ax[0].set_title("training loss")
ax[1].set_title("validation new-burn IoU (pooled)")
for a in ax:
    a.legend()
    a.set_xlabel("epoch")
plt.tight_layout()
plt.savefig(f"{OUT}/training_curves.png", dpi=150)

In [ ]:
# predictions for the largest steps of each test fire
def load_model(mode):
    ck = torch.load(f"{OUT}/{mode}.pt", map_location="cuda", weights_only=False)
    m = ml.FireModel(ck["normalizer"]["channels"], mode, base=ck["config"]["base"]).cuda()
    m.load_state_dict(ck["state_dict"])
    norm = ml.Normalizer(
        ck["normalizer"]["channels"],
        np.array(ck["normalizer"]["mean"], "float32"),
        np.array(ck["normalizer"]["std"], "float32"),
    )
    return m, norm


test = [f for f in fires if f.split == "test"]
modes = list(results)
for f in test:
    bt = np.nan_to_num(f.x[:, f.channels.index("burned_t")])
    grow = [np.nansum(f.y[s] - bt[s]) for s in range(len(bt))]
    steps = np.argsort(grow)[::-1][:3]
    preds = {}
    for m in modes:
        net, norm = load_model(m)
        preds[m] = ml.predict_fire(net, f, norm.apply(f), "cuda")
    fig, axes = plt.subplots(
        len(steps), len(modes) + 1, figsize=(4 * (len(modes) + 1), 3.6 * len(steps)), squeeze=False
    )
    for r, s in enumerate(steps):
        axes[r, 0].imshow(np.nan_to_num(f.y[s]) + bt[s], cmap="OrRd", vmin=0, vmax=2)
        axes[r, 0].set_title(f"fire {f.fire_id} step {s}: observed (dark = burned at t)")
        for c, m in enumerate(modes, 1):
            axes[r, c].imshow(preds[m][s], cmap="OrRd", vmin=0, vmax=1)
            axes[r, c].contour(np.nan_to_num(f.y[s]), [0.5], colors="blue", linewidths=0.7)
            axes[r, c].set_title(f"{m} (blue = observed t+1)")
    for a in axes.ravel():
        a.set_xticks([])
        a.set_yticks([])
    plt.tight_layout()
    plt.savefig(f"{OUT}/test_{f.fire_id}_examples.png", dpi=120)
    plt.show()

In [ ]:
json.dump(
    {m: {k: v for k, v in r.items() if k != "history"} for m, r in results.items()},
    open(f"{OUT}/all_results.json", "w"),
    indent=2,
    default=float,
)
print("saved to", OUT)